In [3]:
import pandas as pd 

data = pd.read_csv('/Users/aimlock/Desktop/python/E_Commerce_ETL_Project/Dataset/raw/Online_Retail_Data_Set.csv', encoding='latin1')
print(data.head())


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  01-12-2010 08:26       2.55     17850.0  United Kingdom  
1  01-12-2010 08:26       3.39     17850.0  United Kingdom  
2  01-12-2010 08:26       2.75     17850.0  United Kingdom  
3  01-12-2010 08:26       3.39     17850.0  United Kingdom  
4  01-12-2010 08:26       3.39     17850.0  United Kingdom  


In [4]:
#TRANSFORM 
pd.isna(data).sum()


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [5]:
#Missing Values Fixed
data = data.dropna(subset=['CustomerID'])


In [6]:
#Negative Quantity Fixed
data = data[data['Quantity'] > 0 ]



In [7]:
#Finding Duplicates values and removing them 
data = data.drop_duplicates()



In [8]:
#Date Formatting
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], format='%d-%m-%Y %H:%M')
# data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], dayfirst=True)
print(data.InvoiceDate.head())

0   2010-12-01 08:26:00
1   2010-12-01 08:26:00
2   2010-12-01 08:26:00
3   2010-12-01 08:26:00
4   2010-12-01 08:26:00
Name: InvoiceDate, dtype: datetime64[ns]


In [9]:
#Feature Engineering
data['Revenue'] = data['UnitPrice'] * data['Quantity'] #Revenue = UnitPrice * Quantity
data['Month'] = data['InvoiceDate'].dt.to_period('M') #Extracting month from InvoiceDate and creating a new column 'Month'

customer_spend = data.groupby('CustomerID')['Revenue'].sum().reset_index() #Grouping by CustomerID and summing the Revenue for each customer
customer_spend.rename(columns={'Revenue': 'TotalSpend'}, inplace=True)

order_count = data.groupby('CustomerID')['InvoiceNo'].nunique().reset_index() #Grouping by CustomerID and counting the unique InvoiceNo for each customer to get the number of orders
order_count.rename(columns={'InvoiceNo': 'OrderCount'}, inplace=True)

Customer_Features = customer_spend.merge(order_count, on='CustomerID') #Merging the customer_spend and order_count dataframes on CustomerID to create a new dataframe Customer_Features that contains TotalSpend and OrderCount for each customer

print(Customer_Features.head())


   CustomerID  TotalSpend  OrderCount
0     12346.0    77183.60           1
1     12347.0     4310.00           7
2     12348.0     1797.24           4
3     12349.0     1757.55           1
4     12350.0      334.40           1


In [10]:
data.to_csv('/Users/aimlock/Desktop/python/E_Commerce_ETL_Project/Dataset/processed/cleaned_data.csv', index=False)


In [14]:
print(data.head())
# print(data.duplicated())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  Revenue    Month  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom    15.30  2010-12  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom    20.34  2010-12  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom    22.00  2010-12  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom    20.34  2010-12  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom    20.34  2010-12  
